# Strategic Compensation & Pay Equity Analytics

## 02 — Data Quality Audit

### Objective

Evaluate the completeness, accuracy, consistency, uniqueness, and referential integrity of six employee compensation datasets before data cleaning and feature engineering.

### Scope

The audit will examine:

- Missing values and completeness
- Duplicate records and primary key integrity
- Foreign key and cross-table consistency
- Data types and categorical validity
- Compensation and salary-band validity
- Performance and promotion consistency
- Employment and reporting date anomalies
- Potential numerical outliers
- Data quality issue severity and proposed treatment

### Methodology

Perform systematic data quality checks using Python and Pandas. Record identified issues, affected record counts, and recommended corrective actions.



In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


In [2]:
##Define Project Paths


project_path = Path(r"C:\Users\Admin\OneDrive\Documents\Compensation & Pay Equity Analysis")

data_path = project_path / "data"

print("Data Folder Exists:", data_path.exists())


Data Folder Exists: True


In [3]:
##Load All Six Datasets


employees = pd.read_csv(data_path / "employees.csv")
jobs = pd.read_csv(data_path / "jobs.csv")
compensation = pd.read_csv(data_path / "compensation.csv")
salary_bands = pd.read_csv(data_path / "salary_bands.csv")
performance = pd.read_csv(data_path / "performance.csv")
promotions = pd.read_csv(data_path / "promotions.csv")

datasets = {
    "Employees": employees,
    "Jobs": jobs,
    "Compensation": compensation,
    "Salary Bands": salary_bands,
    "Performance": performance,
    "Promotions": promotions
}

print("All six raw datasets loaded successfully.")


All six raw datasets loaded successfully.


In [4]:
## Verify Dataset Loading


dataset_counts = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns)
    }
    for name, df in datasets.items()
])

display(dataset_counts)



,Dataset,Rows,Columns
0,Employees,10000,11
1,Jobs,50,7
2,Compensation,30018,7
3,Salary Bands,300,7
4,Performance,30000,4
5,Promotions,2737,5


In [5]:
## Dataset Completeness Summary


completeness_summary = []

for name, df in datasets.items():

    total_cells = df.size
    missing_cells = df.isnull().sum().sum()

    completeness_pct = (
        (total_cells - missing_cells) / total_cells
    ) * 100

    completeness_summary.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Total_Columns": len(df.columns),
        "Total_Cells": total_cells,
        "Missing_Cells": missing_cells,
        "Completeness_Pct": round(completeness_pct, 2)
    })

completeness_df = pd.DataFrame(completeness_summary)

display(completeness_df)


,Dataset,Total_Rows,Total_Columns,Total_Cells,Missing_Cells,Completeness_Pct
0,Employees,10000,11,110000,70,99.94
1,Jobs,50,7,350,0,100.00
2,Compensation,30018,7,210126,75,99.96
3,Salary Bands,300,7,2100,0,100.00
4,Performance,30000,4,120000,60,99.95
5,Promotions,2737,5,13685,0,100.00


### Dataset Completeness Audit

**Objective:** Measure the proportion of available data across all six datasets.

**Formula:**

Completeness (%) = ((Total Cells − Missing Cells) / Total Cells) × 100

**Key Findings:**
- Employees: 70 missing cells (99.94% completeness).
- Compensation: 75 missing cells (99.96% completeness).
- Performance: 60 missing cells (99.95% completeness).
- Jobs, Salary Bands, and Promotions: 100% completeness.

**Business Interpretation:**

All six datasets demonstrate high overall completeness. However, missing values in employee, compensation, and performance attributes could affect workforce reporting, pay equity comparisons, and performance-based compensation analysis.

**Next Action:** Perform column-level missing value analysis to identify affected attributes and assess their business importance.

In [6]:
## Column-Level Missing Values


missing_summary = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isnull().sum()

        if missing_count > 0:

            missing_pct = (
                missing_count / len(df)
            ) * 100

            missing_summary.append({
                "Dataset": name,
                "Column": column,
                "Missing_Count": missing_count,
                "Missing_Pct": round(missing_pct, 2)
            })

missing_df = pd.DataFrame(missing_summary)

display(missing_df)


,Dataset,Column,Missing_Count,Missing_Pct
0,Employees,Education_Level,45,0.45
1,Employees,Years_Experience,25,0.25
2,Compensation,Annual_Bonus,75,0.25
3,Performance,Performance_Rating,60,0.20


### Column-Level Missing Value Analysis

**Formula:**

Missing Percentage (%) = (Missing Values / Total Records) × 100

**Key Findings:**
- Education_Level: 45 missing records (0.45%).
- Years_Experience: 25 missing records (0.25%).
- Annual_Bonus: 75 missing records (0.25%).
- Performance_Rating: 60 missing records (0.20%).

**Business Impact:**

Missing education and experience information may affect workforce segmentation and compensation comparisons. Missing bonus values may affect total compensation calculations, while missing performance ratings may limit performance-based pay analysis.

**Audit Decision:** Retain all original records and document the missing values. Appropriate treatment will be determined during the Data Cleaning phase.

In [7]:
## Exact Duplicate Detection


duplicate_summary = []

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    duplicate_pct = (
        duplicate_count / len(df)
    ) * 100

    duplicate_summary.append({
        "Dataset": name,
        "Total_Records": len(df),
        "Exact_Duplicates": duplicate_count,
        "Duplicate_Pct": round(duplicate_pct, 2)
    })

duplicate_df = pd.DataFrame(duplicate_summary)

display(duplicate_df)


,Dataset,Total_Records,Exact_Duplicates,Duplicate_Pct
0,Employees,10000,0,0.00
1,Jobs,50,0,0.00
2,Compensation,30018,18,0.06
3,Salary Bands,300,0,0.00
4,Performance,30000,0,0.00
5,Promotions,2737,0,0.00


### Exact Duplicate Records Audit

**Formula:**

Duplicate Percentage (%) = (Exact Duplicate Records / Total Records) × 100

**Key Findings:**
- Compensation contains 18 exact duplicate records (0.06%).
- The remaining five datasets contain no exact duplicate records.

**Business Impact:**

Duplicate compensation records may inflate compensation expenditure and distort workforce compensation KPIs when records are aggregated.

**Audit Decision:**

Flag the 18 duplicate compensation records for investigation. Duplicate removal, where justified, will be performed in the Data Cleaning phase.

In [8]:
## Business Key Uniqueness

key_columns = {
    "Employees": ["Employee_ID"],
    "Jobs": ["Job_ID"],
    "Compensation": ["Employee_ID", "Compensation_Year"],
    "Salary Bands": ["Job_ID", "Location"],
    "Performance": ["Employee_ID", "Review_Year"],
    "Promotions": ["Employee_ID", "Promotion_Date"]
}

key_audit = []

for name, columns in key_columns.items():

    df = datasets[name]

    duplicate_keys = df.duplicated(
        subset=columns
    ).sum()

    missing_keys = df[columns].isnull().any(axis=1).sum()

    key_audit.append({
        "Dataset": name,
        "Key_Columns": ", ".join(columns),
        "Total_Records": len(df),
        "Duplicate_Keys": duplicate_keys,
        "Missing_Key_Records": missing_keys,
        "Key_Unique": duplicate_keys == 0 and missing_keys == 0
    })

key_audit_df = pd.DataFrame(key_audit)

display(key_audit_df)


,Dataset,Key_Columns,Total_Records,Duplicate_Keys,Missing_Key_Records,Key_Unique
0,Employees,Employee_ID,10000,0,0,True
1,Jobs,Job_ID,50,0,0,True
2,Compensation,"Employee_ID, Compensation_Year",30018,18,0,False
3,Salary Bands,"Job_ID, Location",300,0,0,True
4,Performance,"Employee_ID, Review_Year",30000,0,0,True
5,Promotions,"Employee_ID, Promotion_Date",2737,0,0,True


### Primary Key and Composite Key Integrity Audit

**Objective:** Validate the uniqueness and completeness of candidate primary keys across all six datasets.

**Key Findings:**
- Employees, Jobs, Salary Bands, Performance, and Promotions passed the candidate key uniqueness and completeness checks.
- Compensation contains 18 duplicate Employee_ID–Compensation_Year combinations.
- No missing key values were identified in any dataset.

**Business Impact:**

Duplicate employee-year compensation records may result in overstated compensation expenditure and inaccurate salary analysis.

**Audit Decision:**

Flag the 18 duplicate compensation records for correction during the Data Cleaning phase. Retain all original records during the audit.

In [9]:
## Foreign Key Integrity Audit


foreign_key_checks = [
    ("Employees", "Job_ID", "Jobs", "Job_ID"),
    ("Compensation", "Employee_ID", "Employees", "Employee_ID"),
    ("Performance", "Employee_ID", "Employees", "Employee_ID"),
    ("Promotions", "Employee_ID", "Employees", "Employee_ID"),
    ("Salary Bands", "Job_ID", "Jobs", "Job_ID")
]

fk_results = []

for child_name, child_key, parent_name, parent_key in foreign_key_checks:

    child_df = datasets[child_name]
    parent_df = datasets[parent_name]

    invalid_mask = (
        child_df[child_key].notna()
        & ~child_df[child_key].isin(parent_df[parent_key])
    )

    invalid_count = invalid_mask.sum()
    missing_count = child_df[child_key].isna().sum()

    fk_results.append({
        "Child_Dataset": child_name,
        "Foreign_Key": child_key,
        "Parent_Dataset": parent_name,
        "Total_Records": len(child_df),
        "Invalid_References": invalid_count,
        "Missing_References": missing_count,
        "FK_Valid": invalid_count == 0 and missing_count == 0
    })

fk_audit_df = pd.DataFrame(fk_results)

display(fk_audit_df)


,Child_Dataset,Foreign_Key,Parent_Dataset,Total_Records,Invalid_References,Missing_References,FK_Valid
0,Employees,Job_ID,Jobs,10000,0,0,True
1,Compensation,Employee_ID,Employees,30018,0,0,True
2,Performance,Employee_ID,Employees,30000,0,0,True
3,Promotions,Employee_ID,Employees,2737,0,0,True
4,Salary Bands,Job_ID,Jobs,300,0,0,True


### Foreign Key Integrity Audit

**Objective:** Validate referential integrity across employee, job, compensation, performance, promotion, and salary-band datasets.

**Key Findings:**
- All five foreign-key relationships passed validation.
- No invalid foreign-key references were identified.
- No missing foreign-key values were found.

**Business Interpretation:**

The datasets demonstrate strong referential integrity at the identifier level, supporting reliable relational joins for workforce and compensation analysis.

**Audit Decision:**

No foreign-key corrections are required for the relationships tested. Composite relationships and historical consistency will be examined separately.

In [ ]:
## Validate Employee Salary Band Matching

employee_band_check = employees.merge(
    salary_bands[["Job_ID", "Location"]],
    on=["Job_ID", "Location"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

band_summary = pd.DataFrame({
    "Check": [
        "Total Employees",
        "Employees With Matching Salary Band",
        "Employees Without Matching Salary Band",
        "Salary Band Match Rate (%)"
    ],
    "Result": [
        len(employee_band_check),
        (employee_band_check["_merge"] == "both").sum(),
        (employee_band_check["_merge"] == "left_only").sum(),
        round(
            (employee_band_check["_merge"] == "both").mean() * 100,
            2
        )
    ]
})

display(band_summary)


,Check,Result
0,Total Employees,10000.0
1,Employees With Matching Salary Band,10000.0
2,Employees Without Matching Salary Band,0.0
3,Salary Band Match Rate (%),100.0


### Employee Salary Band Relationship Audit

**Objective:** Validate employee salary-band coverage using Job_ID and Location.

**Formula:**

Salary Band Match Rate (%) = (Employees with Matching Salary Bands / Total Employees) × 100

**Key Findings:**
- All 10,000 employees have matching salary-band records.
- No unmatched employee job-location combinations were identified.
- Salary-band match rate: 100%.

**Business Interpretation:**

Complete salary-band coverage supports consistent salary benchmarking and compensation analysis across jobs and locations.

**Audit Decision:**

No missing salary-band relationships require correction. Salary values and salary-band boundaries will be validated separately.

In [11]:
### Categorical Value Audit


categorical_audit = []

for name, df in datasets.items():

    categorical_columns = df.select_dtypes(
        include=["object", "string", "category"]
    ).columns

    for column in categorical_columns:

        values = df[column].dropna().astype(str).str.strip()

        categorical_audit.append({
            "Dataset": name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Unique_Values": values.nunique(),
            "Blank_Strings": values.eq("").sum(),
            "Sample_Values": ", ".join(
                values.unique()[:5]
            )
        })

categorical_audit_df = pd.DataFrame(categorical_audit)

display(categorical_audit_df)


,Dataset,Column,Data_Type,Unique_Values,Blank_Strings,Sample_Values
0,Employees,Employee_ID,str,10000,0,"E00001, E00002, E00003, E00004, E00005"
1,Employees,Gender,str,3,0,"Male, Female, Non-binary"
2,Employees,Education_Level,str,5,0,"Master's, Doctorate, Bachelor's, MBA, Diploma"
3,Employees,Location,str,6,0,"Chennai, Bengaluru, Hyderabad, Gurugram, Pune"
4,Employees,Hire_Date,str,4601,0,"2025-02-16, 2020-07-28, 2024-05-20, 2026-09-19..."
5,Employees,Job_ID,str,50,0,"J038, J003, J040, J008, J019"
6,Employees,Employment_Type,str,2,0,"Full-time, Part-time"
7,Employees,Employment_Status,str,2,0,"Active, Exited"
8,Jobs,Job_ID,str,50,0,"J001, J002, J003, J004, J005"
9,Jobs,Job_Title,str,50,0,"HR Coordinator, HR Analyst, Senior HR Analyst,..."


### Data Type and Categorical Value Inspection

**Objective:** Examine categorical attributes, text completeness, and date storage formats.

**Key Findings:**
- No blank strings were identified in the inspected text columns.
- Gender contains 3 categories, employment type contains 2, and employment status contains 2.
- Employee locations contain 6 distinct values.
- Compensation and salary-band currencies are consistently recorded as INR.
- Hire_Date and Promotion_Date are stored as text and require date conversion.
- Category validity requires comparison against approved business values.

**Business Interpretation:**

Consistent categorical values support workforce segmentation, compensation benchmarking, and reliable Power BI filtering. Proper date types are essential for tenure calculations, promotion timelines, and historical analysis.

**Audit Decision:**

Retain original values during the audit. Validate date formats and approved categories before determining any required cleaning actions.

In [18]:

numerical_rules = {
    "Employees": {
        "Age": "positive",
        "Years_Experience": "non_negative",
        "Tenure_Years_2026": "non_negative"
    },
    "Compensation": {
        "Base_Salary": "positive",
        "Annual_Bonus": "non_negative",
        "Allowances": "non_negative",
        "Total_Compensation": "positive"
    },
    "Salary Bands": {
        "Band_Min": "positive",
        "Band_Mid": "positive",
        "Band_Max": "positive"
    },
    "Performance": {
        "Performance_Rating": "positive"
    }
}

numerical_results = []

for name, rules in numerical_rules.items():

    df = datasets[name]

    for column, rule in rules.items():

        values = pd.to_numeric(df[column], errors="coerce")

        if rule == "positive":
            invalid_count = (values <= 0).sum()
        else:
            invalid_count = (values < 0).sum()

        numerical_results.append({
            "Dataset": name,
            "Column": column,
            "Rule": rule,
            "Invalid_Values": int(invalid_count),
            "Missing_Values": int(values.isna().sum()),
            "Minimum_Value": values.min(),
            "Maximum_Value": values.max()
        })

numerical_audit_df = pd.DataFrame(numerical_results)

display(numerical_audit_df)


,Dataset,Column,Rule,Invalid_Values,Missing_Values,Minimum_Value,Maximum_Value
0,Employees,Age,positive,0,0,21.00,60.00
1,Employees,Years_Experience,non_negative,0,25,0.00,39.00
2,Employees,Tenure_Years_2026,non_negative,0,0,0.00,21.00
3,Compensation,Base_Salary,positive,0,0,278807.00,6750554.67
4,Compensation,Annual_Bonus,non_negative,0,75,0.00,1604680.15
5,Compensation,Allowances,non_negative,0,0,4514.40,348111.29
6,Compensation,Total_Compensation,positive,0,0,289398.31,8544911.42
7,Salary Bands,Band_Min,positive,0,0,295620.00,4368000.00
8,Salary Bands,Band_Mid,positive,0,0,379000.00,5600000.00
9,Salary Bands,Band_Max,positive,0,0,473750.00,7000000.00


### Numerical Value and Business Rule Validation

**Objective:** Validate numerical fields against positive and non-negative business rules.

**Validation Rules:**
- Positive values must be greater than zero.
- Non-negative values must be greater than or equal to zero.
- Missing values are reported separately from invalid numerical values.

**Key Findings:**
- All 12 numerical fields passed the positive and non-negative validation checks.
- No invalid negative or zero values were identified under the applicable rules.
- Missing values remain in Years_Experience (25), Annual_Bonus (75), and Performance_Rating (60).
- Employee ages range from 21 to 60 years.
- Employee tenure ranges from 0 to 21 years.
- Recorded performance ratings range from 1 to 5.

**Business Interpretation:**

The numerical fields demonstrate consistency with the initial business rules. Further validation is required for compensation calculations, salary-band boundaries, and relationships between numerical attributes.

**Audit Decision:**

No corrections are required for the numerical values tested. Missing values and additional business-rule checks will be addressed separately.

In [19]:
## Compensation Reconciliation


compensation_check = compensation.copy()

compensation_check["Calculated_Total"] = (
    compensation_check["Base_Salary"]
    + compensation_check["Annual_Bonus"]
    + compensation_check["Allowances"]
)

compensation_check["Difference"] = (
    compensation_check["Total_Compensation"]
    - compensation_check["Calculated_Total"]
)

valid_records = compensation_check["Calculated_Total"].notna()

mismatch_mask = (
    valid_records
    & (compensation_check["Difference"].abs() > 0.01)
)

reconciliation_summary = pd.DataFrame({
    "Check": [
        "Total Compensation Records",
        "Records With Complete Components",
        "Records With Missing Components",
        "Compensation Mismatches",
        "Reconciliation Rate (%)"
    ],
    "Result": [
        len(compensation_check),
        valid_records.sum(),
        (~valid_records).sum(),
        mismatch_mask.sum(),
        round(
            (valid_records.sum() - mismatch_mask.sum())
            / valid_records.sum() * 100,
            2
        ) if valid_records.sum() > 0 else np.nan
    ]
})

display(reconciliation_summary)


,Check,Result
0,Total Compensation Records,30018.00
1,Records With Complete Components,29943.00
2,Records With Missing Components,75.00
3,Compensation Mismatches,6619.00
4,Reconciliation Rate (%),77.89


### Compensation Reconciliation Audit

**Objective:** Validate the consistency of recorded total compensation against its individual components.

**Business Formula:**

Total Compensation = Base Salary + Annual Bonus + Allowances

**Validation Rule:**

A reconciliation mismatch is identified when the absolute difference between recorded and calculated compensation exceeds ₹0.01.

**Key Findings:**
- Total compensation records: 30,018.
- Records with complete compensation components: 29,943.
- Records with missing components: 75.
- Compensation mismatches: 6,619.
- Reconciliation rate: 77.89%.

**Business Interpretation:**

Approximately 22.11% of complete compensation records fail the reconciliation check. These discrepancies could affect compensation expenditure reporting and pay equity analysis.

**Audit Decision:**

Flag mismatched records for investigation. Verify the source of the discrepancies and confirm the applicable compensation calculation rule before making corrections during the Data Cleaning phase.

In [20]:
## Investigate Compensation Mismatches


mismatched_records = compensation_check.loc[
    mismatch_mask
].copy()

difference_summary = pd.DataFrame({
    "Metric": [
        "Total Mismatched Records",
        "Overstated Compensation Records",
        "Understated Compensation Records",
        "Average Absolute Difference",
        "Median Absolute Difference",
        "Maximum Absolute Difference",
        "Total Absolute Difference"
    ],
    "Value": [
        len(mismatched_records),
        (mismatched_records["Difference"] > 0).sum(),
        (mismatched_records["Difference"] < 0).sum(),
        mismatched_records["Difference"].abs().mean(),
        mismatched_records["Difference"].abs().median(),
        mismatched_records["Difference"].abs().max(),
        mismatched_records["Difference"].abs().sum()
    ]
})

display(difference_summary.round(2))


,Metric,Value
0,Total Mismatched Records,6619.00
1,Overstated Compensation Records,3339.00
2,Understated Compensation Records,3280.00
3,Average Absolute Difference,0.01
4,Median Absolute Difference,0.01
5,Maximum Absolute Difference,0.01
6,Total Absolute Difference,66.19


In [22]:
## Final Verification: Rounding Differences

difference_values = compensation_check["Difference"].abs()

precision_check = pd.DataFrame({
    "Metric": [
        "Maximum Difference (6 decimals)",
        "Differences Above ₹0.01",
        "Differences Above ₹0.02",
        "Differences Above ₹1.00",
        "Differences Above ₹100.00"
    ],
    "Value": [
        round(difference_values.max(), 6),
        int((difference_values > 0.010001).sum()),
        int((difference_values > 0.02).sum()),
        int((difference_values > 1).sum()),
        int((difference_values > 100).sum())
    ]
})

display(precision_check)


,Metric,Value
0,Maximum Difference (6 decimals),0.01
1,Differences Above ₹0.01,0.00
2,Differences Above ₹0.02,0.00
3,Differences Above ₹1.00,0.00
4,Differences Above ₹100.00,0.00


### Compensation Difference Investigation

**Objective:** Investigate the magnitude and direction of differences between reported and calculated total compensation.

**Key Findings:**
- Initial reconciliation check flagged 6,619 records.
- 3,339 records had reported compensation above the calculated amount.
- 3,280 records had reported compensation below the calculated amount.
- The maximum absolute difference was approximately ₹0.01.
- No records had differences exceeding ₹0.010001.

**Business Interpretation:**

The identified discrepancies are consistent with minor rounding or floating-point precision differences rather than material compensation inconsistencies.

**Audit Decision:**

Apply a small numerical tolerance during reconciliation validation. Retain the 75 records with missing compensation components for further investigation during Data Cleaning.

No compensation values were modified during this audit.

In [23]:
## Salary Band Boundary Audit


band_check = salary_bands.copy()

band_check["Invalid_Band_Order"] = (
    (band_check["Band_Min"] > band_check["Band_Mid"])
    | (band_check["Band_Mid"] > band_check["Band_Max"])
)

band_check["Invalid_Band_Values"] = (
    (band_check["Band_Min"] <= 0)
    | (band_check["Band_Mid"] <= 0)
    | (band_check["Band_Max"] <= 0)
)

band_summary = pd.DataFrame({
    "Check": [
        "Total Salary Bands",
        "Invalid Band Order",
        "Invalid Band Values",
        "Valid Salary Bands",
        "Band Validation Rate (%)"
    ],
    "Result": [
        len(band_check),
        band_check["Invalid_Band_Order"].sum(),
        band_check["Invalid_Band_Values"].sum(),
        (
            ~(
                band_check["Invalid_Band_Order"]
                | band_check["Invalid_Band_Values"]
            )
        ).sum(),
        round(
            (
                ~(
                    band_check["Invalid_Band_Order"]
                    | band_check["Invalid_Band_Values"]
                )
            ).mean() * 100,
            2
        )
    ]
})

display(band_summary)


,Check,Result
0,Total Salary Bands,300.0
1,Invalid Band Order,0.0
2,Invalid Band Values,0.0
3,Valid Salary Bands,300.0
4,Band Validation Rate (%),100.0


### Salary Band Boundary Validation

**Objective:** Validate the numerical boundaries and ordering of salary-band definitions.

**Business Rules:**

- Band_Min ≤ Band_Mid ≤ Band_Max
- All salary-band values must be positive.

**Key Findings:**
- Total salary bands evaluated: 300.
- Invalid band ordering: 0.
- Invalid salary-band values: 0.
- Salary-band validation rate: 100%.

**Business Interpretation:**

All salary bands have valid numerical boundaries, supporting consistent salary benchmarking and compensation analysis across jobs and locations.

**Audit Decision:**

No salary-band boundary corrections are required. Employee compensation will be compared against these reference bands in a subsequent analysis.

In [25]:
## Salary Band Compliance


salary_compliance = compensation.merge(
    employees[["Employee_ID", "Job_ID", "Location"]],
    on="Employee_ID",
    how="left",
    validate="many_to_one"
)

salary_compliance = salary_compliance.merge(
    salary_bands[["Job_ID", "Location", "Band_Min", "Band_Max"]],
    on=["Job_ID", "Location"],
    how="left",
    validate="many_to_one"
)

salary_compliance["Band_Status"] = np.select(
    [
        salary_compliance["Base_Salary"] < salary_compliance["Band_Min"],
        salary_compliance["Base_Salary"] > salary_compliance["Band_Max"],
        salary_compliance["Base_Salary"].between(
            salary_compliance["Band_Min"],
            salary_compliance["Band_Max"]
        )
    ],
    [
        "Below Band",
        "Above Band",
        "Within Band"
    ],
    default="Unmatched"
)

band_compliance_summary = (
    salary_compliance["Band_Status"]
    .value_counts()
    .rename_axis("Band_Status")
    .reset_index(name="Records")
)

band_compliance_summary["Percentage"] = (
    band_compliance_summary["Records"]
    / len(salary_compliance) * 100
).round(2)

display(band_compliance_summary)


,Band_Status,Records,Percentage
0,Within Band,29679,98.87
1,Below Band,338,1.13
2,Above Band,1,0.00


### Employee Salary Band Compliance Audit

**Objective:** Evaluate whether employee base salaries fall within the salary bands associated with their recorded jobs and locations.

**Business Rules:**
- Below Band: Base_Salary < Band_Min
- Within Band: Band_Min ≤ Base_Salary ≤ Band_Max
- Above Band: Base_Salary > Band_Max

**Key Findings:**
- Within Band: 29,679 records (98.87%).
- Below Band: 338 records (1.13%).
- Above Band: 1 record (less than 0.01%).
- Total potential salary-band exceptions: 339 records.

**Business Interpretation:**

Most compensation records align with their assigned salary bands. Below-band and above-band exceptions should be reviewed to determine whether they reflect compensation policy deviations or differences in historical job assignments and salary structures.

**Audit Limitation:**

Historical compensation is compared with currently recorded employee jobs, locations, and salary bands. These results are preliminary and do not independently establish pay inequity.

**Audit Decision:**

Flag the 339 out-of-band records for further investigation. Do not automatically modify employee salaries or salary-band boundaries.

In [26]:
## Performance Validation


performance_check = performance.copy()

invalid_ratings = (
    performance_check["Performance_Rating"].notna()
    & ~performance_check["Performance_Rating"].between(1, 5)
)

rating_category_check = (
    performance_check
    .groupby("Performance_Rating", dropna=False)["Performance_Category"]
    .nunique(dropna=False)
    .reset_index(name="Unique_Categories")
)

performance_summary = pd.DataFrame({
    "Check": [
        "Total Performance Records",
        "Invalid Ratings",
        "Missing Ratings",
        "Missing Categories",
        "Ratings With Multiple Categories"
    ],
    "Result": [
        len(performance_check),
        invalid_ratings.sum(),
        performance_check["Performance_Rating"].isna().sum(),
        performance_check["Performance_Category"].isna().sum(),
        (
            rating_category_check.loc[
                rating_category_check["Performance_Rating"].notna(),
                "Unique_Categories"
            ] > 1
        ).sum()
    ]
})

display(performance_summary)

display(rating_category_check)


,Check,Result
0,Total Performance Records,30000
1,Invalid Ratings,0
2,Missing Ratings,60
3,Missing Categories,0
4,Ratings With Multiple Categories,0


,Performance_Rating,Unique_Categories
0,1.0,1
1,2.0,1
2,3.0,1
3,4.0,1
4,5.0,1
5,NaN,5


### Performance Rating and Category Consistency Audit

**Objective:** Validate performance rating ranges and consistency between numerical ratings and performance categories.

**Business Rules:**
- Performance ratings must fall between 1 and 5.
- Each valid numerical rating should map consistently to one performance category.

**Key Findings:**
- Total performance records: 30,000.
- Invalid performance ratings: 0.
- Missing performance ratings: 60.
- Missing performance categories: 0.
- Each valid rating maps to one category.
- Records with missing ratings collectively contain five different performance categories.

**Business Interpretation:**

The performance data demonstrates consistent rating-to-category relationships. However, 60 missing numerical ratings may affect performance-based compensation and employee evaluation analysis.

**Audit Decision:**

Retain the original records. Investigate whether the existing performance categories can reliably support missing-rating treatment during the Data Cleaning phase.

In [27]:
## Missing Rating Distribution


missing_rating_records = performance[
    performance["Performance_Rating"].isna()
]

missing_rating_summary = (
    missing_rating_records["Performance_Category"]
    .value_counts(dropna=False)
    .reset_index()
)

missing_rating_summary.columns = [
    "Performance_Category",
    "Missing_Rating_Records"
]

missing_rating_summary["Percentage"] = (
    missing_rating_summary["Missing_Rating_Records"]
    / len(missing_rating_records) * 100
).round(2)

display(missing_rating_summary)


,Performance_Category,Missing_Rating_Records,Percentage
0,Meets Expectations,35,58.33
1,Exceeds Expectations,10,16.67
2,Outstanding,7,11.67
3,Developing,6,10.00
4,Needs Improvement,2,3.33


### Missing Performance Rating Investigation

**Objective:** Examine the distribution of missing numerical performance ratings across performance categories.

**Key Findings:**
- Total missing performance ratings: 60.
- Meets Expectations accounts for 35 missing ratings (58.33%).
- Exceeds Expectations accounts for 10 missing ratings (16.67%).
- Outstanding accounts for 7 missing ratings (11.67%).
- Developing accounts for 6 missing ratings (10.00%).
- Needs Improvement accounts for 2 missing ratings (3.33%).

**Business Interpretation:**

All records with missing numerical ratings retain performance category information. This may provide an opportunity to recover missing ratings if each category consistently corresponds to one numeric rating.

**Audit Decision:**

Investigate category-to-rating consistency before selecting a missing-value treatment. No missing ratings will be filled during the audit.

In [28]:
## Validate Category-to-Rating Mapping


category_rating_mapping = (
    performance.dropna(subset=["Performance_Rating"])
    .groupby("Performance_Category")
    .agg(
        Unique_Ratings=("Performance_Rating", "nunique"),
        Minimum_Rating=("Performance_Rating", "min"),
        Maximum_Rating=("Performance_Rating", "max"),
        Records=("Performance_Rating", "count")
    )
    .reset_index()
    .sort_values("Minimum_Rating")
)

display(category_rating_mapping)


,Performance_Category,Unique_Ratings,Minimum_Rating,Maximum_Rating,Records
3,Needs Improvement,1,1.0,1.0,1236
0,Developing,1,2.0,2.0,3264
2,Meets Expectations,1,3.0,3.0,15519
1,Exceeds Expectations,1,4.0,4.0,8092
4,Outstanding,1,5.0,5.0,1829


### Performance Category-to-Rating Mapping Audit

**Objective:** Validate the consistency of numerical performance ratings and their corresponding performance categories.

**Key Findings:**
- Needs Improvement corresponds to rating 1.
- Developing corresponds to rating 2.
- Meets Expectations corresponds to rating 3.
- Exceeds Expectations corresponds to rating 4.
- Outstanding corresponds to rating 5.
- All five categories have exactly one associated numerical rating.
- The mapping is supported by 29,940 complete performance records.

**Business Interpretation:**

The consistent one-to-one mapping provides a reliable basis for recovering the 60 missing numerical performance ratings using their existing performance categories.

**Audit Decision:**

Recommend category-based missing-rating treatment during the Data Cleaning phase, subject to confirming the organization's rating definitions.

No values were modified during this audit.

In [29]:
## Historical Employment Consistency


employee_hire = employees[[
    "Employee_ID",
    "Hire_Date"
]].copy()

employee_hire["Hire_Year"] = pd.to_datetime(
    employee_hire["Hire_Date"],
    errors="coerce"
).dt.year

# Compensation historical consistency
compensation_hire = compensation.merge(
    employee_hire[["Employee_ID", "Hire_Year"]],
    on="Employee_ID",
    how="left",
    validate="many_to_one"
)

compensation_invalid = (
    compensation_hire["Compensation_Year"]
    < compensation_hire["Hire_Year"]
)

# Performance historical consistency
performance_hire = performance.merge(
    employee_hire[["Employee_ID", "Hire_Year"]],
    on="Employee_ID",
    how="left",
    validate="many_to_one"
)

performance_invalid = (
    performance_hire["Review_Year"]
    < performance_hire["Hire_Year"]
)

historical_audit = pd.DataFrame({
    "Dataset": ["Compensation", "Performance"],
    "Total_Records": [
        len(compensation_hire),
        len(performance_hire)
    ],
    "Records_Before_Hire_Year": [
        int(compensation_invalid.sum()),
        int(performance_invalid.sum())
    ],
    "Invalid_Percentage": [
        round(compensation_invalid.mean() * 100, 2),
        round(performance_invalid.mean() * 100, 2)
    ]
})

display(historical_audit)


,Dataset,Total_Records,Records_Before_Hire_Year,Invalid_Percentage
0,Compensation,30018,3698,12.32
1,Performance,30000,3695,12.32


### Employee Hire Date and Reporting Year Consistency Audit

**Objective:** Identify compensation and performance records associated with reporting years before an employee's hiring year.

**Business Rule:**

Reporting Year ≥ Employee Hire Year

**Key Findings:**
- Compensation: 3,698 records (12.32%) occur before the employee's hiring year.
- Performance: 3,695 records (12.32%) occur before the employee's hiring year.
- Both datasets demonstrate similar levels of historical inconsistency.

**Business Impact:**

Pre-employment compensation and performance records may distort workforce expenditure, employee salary progression, historical performance analysis, and pay equity comparisons.

**Audit Decision:**

Flag these records for further investigation. Determine whether they represent synthetic data generation inconsistencies, incorrect hire dates, or historical record errors before deciding on corrective action.



In [30]:
## Historical Inconsistency Breakdown


# Compensation records before hire year
invalid_compensation = compensation_hire.loc[
    compensation_invalid
].copy()

compensation_year_summary = (
    invalid_compensation
    .groupby("Compensation_Year")
    .agg(
        Invalid_Records=("Employee_ID", "size"),
        Affected_Employees=("Employee_ID", "nunique")
    )
    .reset_index()
)

print("Compensation Historical Inconsistencies")
display(compensation_year_summary)


# Performance records before hire year
invalid_performance = performance_hire.loc[
    performance_invalid
].copy()

performance_year_summary = (
    invalid_performance
    .groupby("Review_Year")
    .agg(
        Invalid_Records=("Employee_ID", "size"),
        Affected_Employees=("Employee_ID", "nunique")
    )
    .reset_index()
)

print("Performance Historical Inconsistencies")
display(performance_year_summary)



Compensation Historical Inconsistencies


,Compensation_Year,Invalid_Records,Affected_Employees
0,2024,2464,2461
1,2025,1234,1234


Performance Historical Inconsistencies


,Review_Year,Invalid_Records,Affected_Employees
0,2024,2461,2461
1,2025,1234,1234


### Historical Inconsistency Investigation by Reporting Year

**Objective:** Identify the reporting years and employees affected by pre-employment compensation and performance records.

**Key Findings:**
- Compensation: 2,464 invalid records in 2024 and 1,234 in 2025.
- Performance: 2,461 invalid records in 2024 and 1,234 in 2025.
- No pre-employment records were identified for 2026.
- The three-record difference in 2024 compensation requires consideration of duplicate records.

**Business Interpretation:**

Historical inconsistencies are concentrated in earlier reporting periods and may distort year-over-year compensation, performance, and workforce analysis.

**Audit Decision:**

Recommend excluding or correcting confirmed pre-employment records during Data Cleaning, based on documented business rules. Preserve the original datasets and maintain an audit trail of all decisions.

In [32]:
## Promotion Date Audit


promotion_check = promotions.merge(
    employees[["Employee_ID", "Hire_Date"]],
    on="Employee_ID",
    how="left",
    validate="many_to_one"
)

promotion_check["Promotion_Date_Parsed"] = pd.to_datetime(
    promotion_check["Promotion_Date"],
    errors="coerce"
)

promotion_check["Hire_Date_Parsed"] = pd.to_datetime(
    promotion_check["Hire_Date"],
    errors="coerce"
)

invalid_promotion_dates = (
    promotion_check["Promotion_Date_Parsed"].isna()
)

invalid_hire_dates = (
    promotion_check["Hire_Date_Parsed"].isna()
)

promotions_before_hire = (
    promotion_check["Promotion_Date_Parsed"]
    < promotion_check["Hire_Date_Parsed"]
)

promotion_date_summary = pd.DataFrame({
    "Check": [
        "Total Promotion Records",
        "Invalid or Missing Promotion Dates",
        "Invalid or Missing Hire Dates",
        "Promotions Before Hire Date",
        "Valid Promotion Records"
    ],
    "Result": [
        len(promotion_check),
        int(invalid_promotion_dates.sum()),
        int(invalid_hire_dates.sum()),
        int(promotions_before_hire.sum()),
        int((
            ~invalid_promotion_dates
            & ~invalid_hire_dates
            & ~promotions_before_hire
        ).sum())
    ]
})

display(promotion_date_summary)


,Check,Result
0,Total Promotion Records,2737
1,Invalid or Missing Promotion Dates,0
2,Invalid or Missing Hire Dates,0
3,Promotions Before Hire Date,132
4,Valid Promotion Records,2605


### Promotion Date and Employee Hire Date Validation

**Objective:** Validate promotion dates against employee hiring dates.

**Business Rule:**

Promotion Date ≥ Hire Date

**Key Findings:**
- Total promotion records: 2,737.
- Invalid or missing promotion dates: 0.
- Invalid or missing employee hire dates: 0.
- Promotions before employee hire dates: 132.
- Valid promotion records: 2,605 (95.18%).

**Business Interpretation:**

Approximately 4.82% of promotion records occur before the associated employee's hiring date. These inconsistencies may distort employee career progression, promotion frequency, and promotion equity analysis.

**Audit Decision:**

Flag the 132 pre-employment promotion records for investigation and appropriate treatment during the Data Cleaning phase. Preserve all original records during auditing.

In [33]:
## Inspect Promotion Numerical Columns


print("Promotion Dataset Columns:")
print(promotions.columns.tolist())

print("\nPromotion Dataset Numerical Summary:")
display(promotions.describe())


Promotion Dataset Columns:
['Employee_ID', 'Promotion_Date', 'Previous_Level', 'New_Level', 'Salary_Increase_Pct']

Promotion Dataset Numerical Summary:


,Previous_Level,New_Level,Salary_Increase_Pct
count,2737.000000,2737.000000,2737.000000
mean,2.852758,3.852758,14.535543
std,1.303625,1.303625,4.292415
min,1.000000,2.000000,7.030000
25%,2.000000,3.000000,10.880000
50%,3.000000,4.000000,14.590000
75%,4.000000,5.000000,18.350000
max,5.000000,6.000000,21.970000


In [34]:
## Promotion Validation


promotion_rules = promotions.copy()

promotion_rules["Invalid_Level_Change"] = (
    promotion_rules["New_Level"]
    <= promotion_rules["Previous_Level"]
)

promotion_rules["Invalid_Level_Range"] = (
    ~promotion_rules["Previous_Level"].between(1, 6)
    | ~promotion_rules["New_Level"].between(1, 6)
)

promotion_rules["Invalid_Salary_Increase"] = (
    promotion_rules["Salary_Increase_Pct"] <= 0
)

promotion_rules["High_Salary_Increase"] = (
    promotion_rules["Salary_Increase_Pct"] > 30
)

promotion_validation = pd.DataFrame({
    "Check": [
        "Total Promotions",
        "Invalid Level Changes",
        "Invalid Level Ranges",
        "Invalid Salary Increases",
        "Salary Increases Above 30%",
        "Average Salary Increase (%)"
    ],
    "Result": [
        len(promotion_rules),
        promotion_rules["Invalid_Level_Change"].sum(),
        promotion_rules["Invalid_Level_Range"].sum(),
        promotion_rules["Invalid_Salary_Increase"].sum(),
        promotion_rules["High_Salary_Increase"].sum(),
        round(
            promotion_rules["Salary_Increase_Pct"].mean(), 2
        )
    ]
})

display(promotion_validation)


,Check,Result
0,Total Promotions,2737.00
1,Invalid Level Changes,0.00
2,Invalid Level Ranges,0.00
3,Invalid Salary Increases,0.00
4,Salary Increases Above 30%,0.00
5,Average Salary Increase (%),14.54


### Promotion Salary Increase and Level Validation

**Objective:** Validate promotion-related job-level changes and salary increase percentages.

**Business Rules:**
- New_Level > Previous_Level.
- Job levels must be between 1 and 6.
- Salary_Increase_Pct must be positive.
- Salary increases above 30% are flagged for review using an illustrative threshold.

**Key Findings:**
- Total promotion records: 2,737.
- Invalid job-level changes: 0.
- Invalid job-level ranges: 0.
- Invalid salary increases: 0.
- Salary increases above 30%: 0.
- Average promotion salary increase: 14.54%.

**Business Interpretation:**

Promotion-related numerical values demonstrate strong consistency with the defined validation rules. All recorded promotions represent upward job-level movements with positive salary increases.

However, 132 promotions were previously identified as occurring before employee hire dates.

**Audit Decision:**

No numerical promotion corrections are required under the tested rules. Pre-employment promotion records remain flagged for investigation during Data Cleaning.

In [ ]:
##Employee Consistency Validation


employee_check = employees.copy()

employee_check["Experience_Age_Issue"] = (
    employee_check["Years_Experience"]
    > employee_check["Age"] - 16
)

employee_check["Tenure_Experience_Issue"] = (
    employee_check["Tenure_Years_2026"]
    > employee_check["Years_Experience"]
)

employee_check["Underage_Employee"] = (
    employee_check["Age"] < 18
)

employee_consistency_summary = pd.DataFrame({
    "Check": [
        "Total Employees",
        "Experience Exceeds Plausible Age Limit",
        "Tenure Exceeds Experience",
        "Employees Below Age 18",
        "Missing Experience Values"
    ],
    "Result": [
        len(employee_check),
        int(employee_check["Experience_Age_Issue"].sum()),
        int(employee_check["Tenure_Experience_Issue"].sum()),
        int(employee_check["Underage_Employee"].sum()),
        int(employee_check["Years_Experience"].isna().sum())
    ]
})

display(employee_consistency_summary)


,Check,Result
0,Total Employees,10000
1,Experience Exceeds Plausible Age Limit,0
2,Tenure Exceeds Experience,2332
3,Employees Below Age 18,0
4,Missing Experience Values,25


### Employee Age, Experience and Tenure Consistency Audit

**Objective:** Evaluate the consistency of employee age, professional experience, and organizational tenure.

**Validation Rules:**
- Years_Experience ≤ Age − 16 (screening assumption).
- Tenure_Years_2026 ≤ Years_Experience (consistency check).
- Age ≥ 18.

**Key Findings:**
- Total employees evaluated: 10,000.
- Experience exceeding the plausible age limit: 0.
- Tenure exceeding professional experience: 2,332 employees (23.32%).
- Employees younger than 18: 0.
- Missing professional experience values: 25.

**Business Interpretation:**

The age-related checks did not identify violations. However, 2,332 employees have organizational tenure exceeding their reported professional experience.

These inconsistencies may affect experience-based compensation benchmarking and workforce segmentation.

**Audit Decision:**

Flag the affected records for investigation. Confirm the definitions and reference dates of experience and tenure before applying any corrections during Data Cleaning.

In [36]:
## Tenure Validation


tenure_check = employees[[
    "Employee_ID",
    "Hire_Date",
    "Tenure_Years_2026"
]].copy()

tenure_check["Hire_Date_Parsed"] = pd.to_datetime(
    tenure_check["Hire_Date"],
    errors="coerce"
)

reference_date = pd.Timestamp("2026-12-31")

tenure_check["Calculated_Tenure"] = (
    reference_date.year
    - tenure_check["Hire_Date_Parsed"].dt.year
    - (
        (tenure_check["Hire_Date_Parsed"].dt.month > reference_date.month)
        | (
            (tenure_check["Hire_Date_Parsed"].dt.month == reference_date.month)
            & (tenure_check["Hire_Date_Parsed"].dt.day > reference_date.day)
        )
    ).astype("Int64")
)

tenure_check["Tenure_Difference"] = (
    tenure_check["Tenure_Years_2026"]
    - tenure_check["Calculated_Tenure"]
)

tenure_summary = pd.DataFrame({
    "Check": [
        "Total Employees",
        "Invalid or Missing Hire Dates",
        "Matching Tenure Records",
        "Tenure Mismatches",
        "Maximum Absolute Tenure Difference"
    ],
    "Result": [
        len(tenure_check),
        int(tenure_check["Hire_Date_Parsed"].isna().sum()),
        int((tenure_check["Tenure_Difference"] == 0).sum()),
        int((tenure_check["Tenure_Difference"].notna()
             & (tenure_check["Tenure_Difference"] != 0)).sum()),
        tenure_check["Tenure_Difference"].abs().max()
    ]
})

display(tenure_summary)


,Check,Result
0,Total Employees,10000.0
1,Invalid or Missing Hire Dates,0.0
2,Matching Tenure Records,493.0
3,Tenure Mismatches,9507.0
4,Maximum Absolute Tenure Difference,1.0


In [37]:
## Investigate Tenure Differences


tenure_difference_summary = (
    tenure_check["Tenure_Difference"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
)

tenure_difference_summary.columns = [
    "Tenure_Difference",
    "Employee_Count"
]

display(tenure_difference_summary)

print("\nSample Tenure Mismatches:")

display(
    tenure_check.loc[
        tenure_check["Tenure_Difference"] != 0,
        [
            "Hire_Date",
            "Tenure_Years_2026",
            "Calculated_Tenure",
            "Tenure_Difference"
        ]
    ].head(10)
)


,Tenure_Difference,Employee_Count
0,0.0,493
1,0.1,548
2,0.1,117
3,0.1,297
4,0.1,56
5,0.2,323
6,0.2,134
7,0.2,119
8,0.2,399
9,0.3,408



Sample Tenure Mismatches:


,Hire_Date,Tenure_Years_2026,Calculated_Tenure,Tenure_Difference
0,2025-02-16,1.9,1,0.9
1,2020-07-28,6.4,6,0.4
2,2024-05-20,2.6,2,0.6
3,2026-09-19,0.3,0,0.3
4,2026-06-07,0.6,0,0.6
5,2021-04-10,5.7,5,0.7
6,2023-04-02,3.7,3,0.7
7,2024-03-15,2.8,2,0.8
8,2008-09-08,18.3,18,0.3
9,2026-02-13,0.9,0,0.9


In [38]:
### Final Tenure Validation


reference_date = pd.Timestamp("2026-12-31")

tenure_check["Calculated_Decimal_Tenure"] = (
    (reference_date - tenure_check["Hire_Date_Parsed"]).dt.days
    / 365.25
).round(1)

tenure_check["Decimal_Difference"] = (
    tenure_check["Tenure_Years_2026"]
    - tenure_check["Calculated_Decimal_Tenure"]
).round(2)

decimal_tenure_summary = pd.DataFrame({
    "Check": [
        "Total Employees",
        "Matching Decimal Tenure",
        "Tenure Differences Above 0.1 Years",
        "Maximum Absolute Difference"
    ],
    "Result": [
        len(tenure_check),
        int((tenure_check["Decimal_Difference"] == 0).sum()),
        int((tenure_check["Decimal_Difference"].abs() > 0.100001).sum()),
        tenure_check["Decimal_Difference"].abs().max()
    ]
})

display(decimal_tenure_summary)


,Check,Result
0,Total Employees,10000.0
1,Matching Decimal Tenure,10000.0
2,Tenure Differences Above 0.1 Years,0.0
3,Maximum Absolute Difference,0.0


### Employee Tenure and Hire Date Consistency Audit

**Objective:** Validate recorded employee tenure against tenure calculated from hiring dates.

**Formula:**

Decimal Tenure = (Reference Date − Hire Date) / 365.25

**Reference Date:** December 31, 2026

**Key Findings:**
- Total employees evaluated: 10,000.
- Matching decimal tenure records: 10,000.
- Tenure mismatches: 0.
- Maximum absolute difference: 0.0 years.
- Tenure consistency rate: 100%.

**Business Interpretation:**

The recorded tenure values are consistent with decimal years of service calculated from employee hire dates.

The initial discrepancies were caused by comparing decimal tenure with completed whole years rather than using a consistent calculation method.

**Audit Decision:**

No tenure corrections are required under the validated calculation method. Retain the existing tenure values.

In [39]:
## Age at Hiring Validation

age_check = employees[[
    "Employee_ID",
    "Age",
    "Hire_Date"
]].copy()

age_check["Hire_Date_Parsed"] = pd.to_datetime(
    age_check["Hire_Date"],
    errors="coerce"
)

reference_date = pd.Timestamp("2026-12-31")

age_check["Years_Since_Hire"] = (
    (reference_date - age_check["Hire_Date_Parsed"]).dt.days
    / 365.25
)

age_check["Estimated_Age_At_Hire"] = (
    age_check["Age"] - age_check["Years_Since_Hire"]
)

age_check["Potential_Underage_Hire"] = (
    age_check["Estimated_Age_At_Hire"] < 18
)

age_audit_summary = pd.DataFrame({
    "Check": [
        "Total Employees",
        "Invalid or Missing Hire Dates",
        "Potential Underage Hires",
        "Minimum Estimated Age at Hire",
        "Average Estimated Age at Hire"
    ],
    "Result": [
        len(age_check),
        int(age_check["Hire_Date_Parsed"].isna().sum()),
        int(age_check["Potential_Underage_Hire"].sum()),
        round(age_check["Estimated_Age_At_Hire"].min(), 2),
        round(age_check["Estimated_Age_At_Hire"].mean(), 2)
    ]
})

display(age_audit_summary)


,Check,Result
0,Total Employees,10000.00
1,Invalid or Missing Hire Dates,0.00
2,Potential Underage Hires,0.00
3,Minimum Estimated Age at Hire,19.01
4,Average Estimated Age at Hire,27.66


### Employee Hire Date and Age Consistency Audit

**Objective:** Validate whether employee ages are logically consistent with their hiring dates.

**Business Rule:**

Estimated Age at Hire = Current Age − Years Since Hire

Employees should have an estimated hiring age of at least 18 years.

**Key Findings:**
- Total employees evaluated: 10,000.
- Invalid or missing hire dates: 0.
- Potential underage hires: 0.
- Minimum estimated age at hire: 19.01 years.
- Average estimated age at hire: 27.66 years.

**Business Interpretation:**

All employee records satisfy the defined minimum hiring-age screening rule. No age-related hiring inconsistencies were identified.

**Audit Decision:**

No corrections are required based on this screening check. Estimated ages are approximate because individual birth dates are unavailable.

In [40]:
## Compensation Outlier Summary

compensation_columns = [
    "Base_Salary",
    "Annual_Bonus",
    "Total_Compensation"
]

outlier_results = []

for column in compensation_columns:

    q1 = compensation[column].quantile(0.25)
    q3 = compensation[column].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (compensation[column] < lower_bound)
        | (compensation[column] > upper_bound)
    ).sum()

    outlier_results.append({
        "Column": column,
        "Lower_Bound": round(lower_bound, 2),
        "Upper_Bound": round(upper_bound, 2),
        "Outlier_Count": int(outlier_count),
        "Outlier_Percentage": round(
            outlier_count / compensation[column].notna().sum() * 100,
            2
        )
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary)



,Column,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,Base_Salary,-1336601.05,4320370.80,4196,13.98
1,Annual_Bonus,-242957.12,601888.66,3265,10.90
2,Total_Compensation,-1605894.69,5038384.80,4179,13.92


In [41]:
## Outlier Distribution by Job Level


salary_outlier_check = compensation.merge(
    employees[["Employee_ID", "Job_ID"]],
    on="Employee_ID",
    how="left"
)

salary_outlier_check = salary_outlier_check.merge(
    jobs[["Job_ID", "Job_Level"]],
    on="Job_ID",
    how="left"
)

q1 = compensation["Base_Salary"].quantile(0.25)
q3 = compensation["Base_Salary"].quantile(0.75)

iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

salary_outlier_check["High_Salary_Outlier"] = (
    salary_outlier_check["Base_Salary"] > upper_bound
)

outliers_by_level = (
    salary_outlier_check
    .groupby("Job_Level", dropna=False)
    .agg(
        Total_Records=("Base_Salary", "size"),
        High_Salary_Outliers=("High_Salary_Outlier", "sum")
    )
    .reset_index()
)

outliers_by_level["Outlier_Percentage"] = (
    outliers_by_level["High_Salary_Outliers"]
    / outliers_by_level["Total_Records"] * 100
).round(2)

display(outliers_by_level)


,Job_Level,Total_Records,High_Salary_Outliers,Outlier_Percentage
0,1,1821,0,0.00
1,2,6578,0,0.00
2,3,6201,0,0.00
3,4,8181,0,0.00
4,5,3017,0,0.00
5,6,4220,4196,99.43


### Compensation Outlier Detection

**Objective:** Identify unusual compensation values and investigate whether they are associated with employee job levels.

**Method:** Interquartile Range (IQR), using 1.5 × IQR to identify potential outliers.

**Key Findings:**
- Base salary outliers: 4,196 (13.98%).
- Annual bonus outliers: 3,265 (10.90%).
- Total compensation outliers: 4,179 (13.92%).
- All 4,196 high base salary outliers belong to Job Level 6.
- 99.43% of Job Level 6 compensation records exceed the overall base salary IQR upper bound.
- No high base salary outliers were identified in Job Levels 1–5.

**Business Interpretation:**

The concentration of high base salary outliers in Job Level 6 indicates that job seniority is a major factor behind the observed salary distribution.

Applying a single IQR threshold across all job levels can incorrectly classify legitimate senior-level compensation as anomalous.

**Audit Decision:**

Retain all compensation records during auditing. Do not automatically remove or cap high salaries. Use job-level and location-aware salary comparisons in subsequent analysis.

**Limitation:** Historical compensation is matched to employees' current job levels, and raw compensation duplicates remain present.

In [42]:
## Salary Band Progression


band_check = salary_bands.copy()

band_check = band_check.sort_values(
    ["Location", "Job_Level", "Job_ID"]
)

band_check["Previous_Band_Mid"] = (
    band_check.groupby(["Location", "Job_ID"])["Band_Mid"]
    .shift(1)
)

band_check["Invalid_Midpoint_Progression"] = (
    band_check["Previous_Band_Mid"].notna()
    & (
        band_check["Band_Mid"]
        <= band_check["Previous_Band_Mid"]
    )
)

band_progression_summary = pd.DataFrame({
    "Check": [
        "Total Salary Band Records",
        "Bands With Previous Level Comparison",
        "Invalid Midpoint Progressions"
    ],
    "Result": [
        len(band_check),
        int(band_check["Previous_Band_Mid"].notna().sum()),
        int(band_check["Invalid_Midpoint_Progression"].sum())
    ]
})

display(band_progression_summary)


,Check,Result
0,Total Salary Band Records,300
1,Bands With Previous Level Comparison,0
2,Invalid Midpoint Progressions,0


In [43]:
## Salary Band Midpoint Progression


# Average salary band midpoint by location and job level

level_band_summary = (
    salary_bands
    .groupby(["Location", "Job_Level"])
    .agg(
        Job_Count=("Job_ID", "nunique"),
        Average_Band_Mid=("Band_Mid", "mean")
    )
    .reset_index()
    .sort_values(["Location", "Job_Level"])
)

# Compare each level with the previous level in the same location

level_band_summary["Previous_Level_Mid"] = (
    level_band_summary
    .groupby("Location")["Average_Band_Mid"]
    .shift(1)
)

level_band_summary["Invalid_Progression"] = (
    level_band_summary["Previous_Level_Mid"].notna()
    & (
        level_band_summary["Average_Band_Mid"]
        <= level_band_summary["Previous_Level_Mid"]
    )
)

display(level_band_summary.round(2))

print(
    "\nTotal Invalid Progressions:",
    level_band_summary["Invalid_Progression"].sum()
)


,Location,Job_Level,Job_Count,Average_Band_Mid,Previous_Level_Mid,Invalid_Progression
0,Bengaluru,1,5,446400.00,NaN,False
1,Bengaluru,2,17,798352.94,446400.00,False
2,Bengaluru,3,14,1261857.14,798352.94,False
3,Bengaluru,4,11,1922818.18,1261857.14,False
4,Bengaluru,5,2,3271000.00,1922818.18,False
5,Bengaluru,6,1,5400000.00,3271000.00,False
6,Chennai,1,5,405000.00,NaN,False
7,Chennai,2,17,724352.94,405000.00,False
8,Chennai,3,14,1145000.00,724352.94,False
9,Chennai,4,11,1744909.09,1145000.00,False



Total Invalid Progressions: 0


### Salary Band Progression Validation

**Objective:** Validate whether average salary band midpoints increase consistently across job levels within each location.

**Method:**
- Calculate the average salary band midpoint by location and job level.
- Compare each level's average midpoint with the preceding level in the same location.
- Flag non-increasing midpoint progression.

**Key Findings:**
- Total salary band records: 300.
- Locations evaluated: 6.
- Job levels per location: 6.
- Valid adjacent-level comparisons: 30.
- Invalid midpoint progressions: 0.

**Business Interpretation:**

Average salary band midpoints increase consistently from Job Level 1 through Job Level 6 in all six locations.

This indicates a logically structured salary hierarchy and supports job-level compensation benchmarking.

**Audit Decision:**

No midpoint progression issues were identified at the location-job-level average. Retain the existing salary band records.

**Limitation:** This validation checks average midpoints rather than individual job-level salary band relationships.

In [44]:
## Annual Record Coverage


# Compensation records by year
comp_years = (
    compensation
    .groupby("Compensation_Year")
    .agg(
        Compensation_Records=("Employee_ID", "size"),
        Employees_With_Compensation=("Employee_ID", "nunique")
    )
    .reset_index()
    .rename(columns={"Compensation_Year": "Year"})
)

# Performance records by year
perf_years = (
    performance
    .groupby("Review_Year")
    .agg(
        Performance_Records=("Employee_ID", "size"),
        Employees_With_Performance=("Employee_ID", "nunique")
    )
    .reset_index()
    .rename(columns={"Review_Year": "Year"})
)

# Combine yearly coverage
year_coverage = comp_years.merge(
    perf_years,
    on="Year",
    how="outer"
).sort_values("Year")

# Compare employee coverage
year_coverage["Employee_Coverage_Difference"] = (
    year_coverage["Employees_With_Compensation"]
    - year_coverage["Employees_With_Performance"]
)

display(year_coverage)


,Year,Compensation_Records,Employees_With_Compensation,Performance_Records,Employees_With_Performance,Employee_Coverage_Difference
0,2024,10006,10000,10000,10000,0
1,2025,10007,10000,10000,10000,0
2,2026,10005,10000,10000,10000,0


In [45]:
## Cross-Dataset Employee-Year Validation


# Unique employee-year combinations

comp_keys = compensation[
    ["Employee_ID", "Compensation_Year"]
].drop_duplicates().rename(
    columns={"Compensation_Year": "Year"}
)

perf_keys = performance[
    ["Employee_ID", "Review_Year"]
].drop_duplicates().rename(
    columns={"Review_Year": "Year"}
)

# Compare employee-year combinations

employee_year_check = comp_keys.merge(
    perf_keys,
    on=["Employee_ID", "Year"],
    how="outer",
    indicator=True
)

employee_year_summary = pd.DataFrame({
    "Check": [
        "Unique Compensation Employee-Year Keys",
        "Unique Performance Employee-Year Keys",
        "Matched Employee-Year Keys",
        "Compensation Keys Without Performance",
        "Performance Keys Without Compensation"
    ],
    "Result": [
        len(comp_keys),
        len(perf_keys),
        int((employee_year_check["_merge"] == "both").sum()),
        int((employee_year_check["_merge"] == "left_only").sum()),
        int((employee_year_check["_merge"] == "right_only").sum())
    ]
})

display(employee_year_summary)


,Check,Result
0,Unique Compensation Employee-Year Keys,30000
1,Unique Performance Employee-Year Keys,30000
2,Matched Employee-Year Keys,30000
3,Compensation Keys Without Performance,0
4,Performance Keys Without Compensation,0


### Cross-Dataset Year Coverage and Employee-Year Validation

**Objective:** Validate reporting-year coverage and employee-year alignment between Compensation and Performance datasets.

**Method:**
- Compare annual record counts and unique employee counts.
- Match distinct Employee_ID and reporting-year combinations across both datasets.

**Key Findings:**
- Reporting years covered: 2024, 2025, and 2026.
- Unique compensation employee-year combinations: 30,000.
- Unique performance employee-year combinations: 30,000.
- Matched employee-year combinations: 30,000 (100%).
- Unmatched employee-year combinations: 0.
- Additional compensation records caused by duplicates: 18.

**Business Interpretation:**

Both datasets provide complete employee-year coverage and can support integrated compensation and performance analysis.

**Audit Decision:**

The employee-year relationship is valid. Remove the 18 confirmed duplicate compensation records during Data Cleaning before joining the datasets.

**Limitation:** Previously identified pre-hire historical records require separate treatment before longitudinal analysis.

In [46]:
# Currency distribution in compensation

comp_currency = (
    compensation["Currency"]
    .value_counts(dropna=False)
    .reset_index()
)

comp_currency.columns = [
    "Currency",
    "Compensation_Records"
]

# Currency distribution in salary bands

band_currency = (
    salary_bands["Currency"]
    .value_counts(dropna=False)
    .reset_index()
)

band_currency.columns = [
    "Currency",
    "Salary_Band_Records"
]

print("Compensation Currency Distribution:")
display(comp_currency)

print("Salary Band Currency Distribution:")
display(band_currency)

# Check for currencies other than INR

currency_summary = pd.DataFrame({
    "Check": [
        "Compensation Records With Non-INR Currency",
        "Salary Bands With Non-INR Currency",
        "Missing Compensation Currency",
        "Missing Salary Band Currency"
    ],
    "Result": [
        int(
            (compensation["Currency"].notna()
             & (compensation["Currency"] != "INR")).sum()
        ),
        int(
            (salary_bands["Currency"].notna()
             & (salary_bands["Currency"] != "INR")).sum()
        ),
        int(compensation["Currency"].isna().sum()),
        int(salary_bands["Currency"].isna().sum())
    ]
})

display(currency_summary)


Compensation Currency Distribution:


,Currency,Compensation_Records
0,INR,30018


Salary Band Currency Distribution:


,Currency,Salary_Band_Records
0,INR,300


,Check,Result
0,Compensation Records With Non-INR Currency,0
1,Salary Bands With Non-INR Currency,0
2,Missing Compensation Currency,0
3,Missing Salary Band Currency,0


### Compensation Currency Consistency Audit

**Objective:** Verify currency consistency across compensation records and salary bands.

**Key Findings:**
- Compensation records evaluated: 30,018.
- Salary band records evaluated: 300.
- Currency used across both datasets: INR.
- Non-INR records: 0.
- Missing currency values: 0.
- Currency consistency rate: 100%.

**Business Interpretation:**

All compensation and salary band amounts use Indian Rupees, allowing direct salary comparisons without currency conversion.

**Audit Decision:**

No currency corrections are required. Retain the existing currency values.

In [47]:
## Promotion-Performance Coverage


# Create a copy for auditing
promotion_perf_check = promotions.copy()

# Extract promotion year
promotion_perf_check["Promotion_Year"] = pd.to_datetime(
    promotion_perf_check["Promotion_Date"],
    errors="coerce"
).dt.year

# Prepare unique performance employee-year keys
performance_keys = (
    performance[["Employee_ID", "Review_Year"]]
    .drop_duplicates()
)

# Match promotion year with performance review year
promotion_perf_check = promotion_perf_check.merge(
    performance_keys,
    left_on=["Employee_ID", "Promotion_Year"],
    right_on=["Employee_ID", "Review_Year"],
    how="left",
    indicator=True
)

promotion_perf_summary = pd.DataFrame({
    "Check": [
        "Total Promotions",
        "Promotions With Same-Year Performance",
        "Promotions Without Same-Year Performance",
        "Invalid Promotion Dates"
    ],
    "Result": [
        len(promotion_perf_check),
        int((promotion_perf_check["_merge"] == "both").sum()),
        int((promotion_perf_check["_merge"] == "left_only").sum()),
        int(promotion_perf_check["Promotion_Year"].isna().sum())
    ]
})

display(promotion_perf_summary)


,Check,Result
0,Total Promotions,2737
1,Promotions With Same-Year Performance,2737
2,Promotions Without Same-Year Performance,0
3,Invalid Promotion Dates,0


### Promotion and Performance Year Alignment Audit

**Objective:** Validate whether promotion records have corresponding employee performance records in the same reporting year.

**Method:**
- Extract the year from Promotion_Date.
- Match promotions with performance records using Employee_ID and reporting year.

**Key Findings:**
- Total promotions: 2,737.
- Promotions with same-year performance records: 2,737 (100%).
- Promotions without matching performance records: 0.
- Invalid promotion dates: 0.

**Business Interpretation:**

All promotions have matching same-year performance records, supporting integrated analysis of employee performance and career progression.

**Audit Decision:**

The promotion-performance year relationship is complete. Retain the records for further analysis, subject to resolution of previously identified date inconsistencies.

**Limitations:**
- 132 promotions occurred before the recorded employee hire dates.
- Same-year alignment does not establish whether performance reviews preceded or influenced promotions.

In [48]:
## Consolidated Data Quality Summary


data_quality_summary = pd.DataFrame({
    "Dataset": [
        "Employees",
        "Employees",
        "Compensation",
        "Compensation",
        "Compensation",
        "Performance",
        "Salary Bands",
        "Promotions",
        "Cross-Dataset"
    ],
    "Quality_Issue": [
        "Missing Education Level",
        "Missing Years of Experience",
        "Duplicate Records",
        "Missing Annual Bonus",
        "Compensation Before Hire Year",
        "Missing Performance Rating",
        "Invalid Salary Band Structure",
        "Promotion Before Hire Date",
        "Unmatched Compensation-Performance Keys"
    ],
    "Affected_Records": [
        45,
        25,
        18,
        75,
        3698,
        60,
        0,
        132,
        0
    ],
    "Recommended_Action": [
        "Review missing categories",
        "Review missing experience",
        "Remove confirmed duplicates",
        "Investigate missing bonus values",
        "Investigate historical eligibility",
        "Validate category-based imputation",
        "No correction required",
        "Investigate invalid promotion dates",
        "No correction required"
    ]
})

display(data_quality_summary)


,Dataset,Quality_Issue,Affected_Records,Recommended_Action
0,Employees,Missing Education Level,45,Review missing categories
1,Employees,Missing Years of Experience,25,Review missing experience
2,Compensation,Duplicate Records,18,Remove confirmed duplicates
3,Compensation,Missing Annual Bonus,75,Investigate missing bonus values
4,Compensation,Compensation Before Hire Year,3698,Investigate historical eligibility
5,Performance,Missing Performance Rating,60,Validate category-based imputation
6,Salary Bands,Invalid Salary Band Structure,0,No correction required
7,Promotions,Promotion Before Hire Date,132,Investigate invalid promotion dates
8,Cross-Dataset,Unmatched Compensation-Performance Keys,0,No correction required


## Final Data Quality Audit Summary

### Objective
Evaluate the completeness, uniqueness, validity, consistency, and relational integrity of the six datasets before data cleaning and SQL analysis.

### Key Data Quality Issues

| Dataset | Issue | Affected Records |
|---|---|---:|
| Employees | Missing Education_Level | 45 |
| Employees | Missing Years_Experience | 25 |
| Employees | Tenure exceeds reported experience | 2,332 |
| Compensation | Exact duplicate records | 18 |
| Compensation | Missing Annual_Bonus | 75 |
| Compensation | Compensation before hire year | 3,698 |
| Performance | Missing Performance_Rating | 60 |
| Performance | Performance review before hire year | 3,695 |
| Promotions | Promotion before hire date | 132 |

### Successful Validations
- Employee and job primary keys are valid.
- Foreign key relationships are complete.
- Compensation and performance employee-year coverage is 100%.
- Employee tenure matches the validated decimal-year calculation.
- Salary band structures and average midpoint progressions are valid.
- Promotion level changes and salary increases satisfy the defined rules.
- Compensation and salary band currencies are consistently INR.

### Important Analytical Considerations

**Compensation outliers:** High salaries are concentrated in senior job levels and should not be automatically removed.

**Compensation reconciliation:** Minor differences of approximately ₹0.01 are consistent with rounding precision.

**Historical eligibility:** Compensation, performance, and promotion records occurring before hire dates require investigation before longitudinal analysis.

**Salary band benchmarking:** Historical compensation is linked to current job and location information because historical job assignments are unavailable.

### Recommended Cleaning Priorities

1. Remove confirmed exact compensation duplicates.
2. Define appropriate treatment for missing employee and compensation values.
3. Validate category-based recovery of missing performance ratings.
4. Investigate and define treatment for pre-hire historical records.
5. Review tenure-versus-experience inconsistencies.
6. Preserve legitimate senior-level compensation values.

### Final Audit Conclusion

The datasets have strong relational integrity and consistent core structures, but several completeness and historical consistency issues require treatment before analysis.

The audit provides a documented foundation for reproducible data cleaning and reliable compensation and pay equity analytics.

No source records were modified during this audit.